# 1. Security Posture Management & Endpoint Strategy

SC-100 asks: **"Design a security posture management strategy that covers hybrid, multicloud, and endpoint environments."**

## Setup

```bash
cd security-certs/sc-100/03-infrastructure
uv sync
# Notebooks use the local .venv directly -- no global kernel to register.
# In VS Code: open the kernel picker (top-right) and select `.venv`.
# In classic Jupyter: uv run jupyter notebook notebooks/
```
Then pick the **`.venv` kernel** for this folder from the VS Code kernel picker (top-right).

If the kernel doesn't appear, reload the VS Code window (`Cmd+Shift+P` → "Reload Window").

## Glossary (read this first if you're new)

Infrastructure security has a lot of acronyms. Here's plain-English meaning for the ones in this notebook.

| Term | What it means (plain English) |
|------|-------------------------------|
| **CSPM** | *Cloud Security Posture Management* — scans your **configurations** ("is this storage account public?") |
| **CWPP** | *Cloud Workload Protection Platform* — watches **running things** for threats (malware, weird processes) |
| **Defender for Cloud** | Microsoft's umbrella product that does both CSPM and CWPP across Azure, AWS, GCP |
| **Defender plan** | A paid CWPP module (Defender for Servers, for Containers, for SQL, …). Turned on per-subscription. |
| **Secure Score** | A 0–100% score for how many security recommendations you've fixed in Defender for Cloud |
| **Attack path** | A chain like *Internet → public VM → managed identity → SQL with PHI*. Defender CSPM draws these; **Microsoft Security Exposure Management** stitches attack paths across Defender for Cloud, Defender XDR and Entra into one enterprise view and is now named in the SC-100 objectives. |
| **Azure Arc** | A tiny agent that makes non-Azure machines (on-prem, AWS EC2, GCP VMs) **look like** Azure resources so Policy/Defender can target them. |
| **EASM** | *External Attack Surface Management* — scanner that finds **your own internet-exposed assets** you may have forgotten about. |
| **EDR** | *Endpoint Detection and Response* — agent on a laptop/server that records activity and flags attacks. |
| **Intune** | Microsoft's MDM/MAM — manages policies on phones and laptops. |
| **LAPS** | *Local Administrator Password Solution* — auto-rotates local admin passwords so every device has a unique one. |
| **IoT / OT** | *Internet of Things / Operational Technology* — MRI machines, PLCs, HVAC. You usually **cannot** install agents on them. |
| **MCSB** | *Microsoft Cloud Security Benchmark* — Microsoft's recommended baseline (shipped as an Azure Policy initiative). |
| **JIT** | *Just-in-Time VM access* — RDP/SSH ports stay closed until a user requests temporary access. |


## Microsoft Defender for Cloud Architecture

Defender for Cloud has two distinct capability sets:

```
┌──────────────────────────────────────────────────────────────────────┐
│                    DEFENDER FOR CLOUD                                │
│                                                                      │
│  ┌────────────────────────────┐  ┌─────────────────────────────────┐ │
│  │       CSPM                 │  │         CWPP (Defender plans)   │ │
│  │  Cloud Security Posture    │  │  Cloud Workload Protection     │ │
│  │  Management                │  │  Platform                      │ │
│  │                            │  │                                 │ │
│  │  FREE tier:                │  │  Defender for Servers (P1/P2)   │ │
│  │  • Secure Score            │  │  Defender for Containers        │ │
│  │  • Basic recommendations   │  │  Defender for App Service       │ │
│  │  • Azure Policy compliance │  │  Defender for Storage           │ │
│  │                            │  │  Defender for Databases         │ │
│  │  PAID (Defender CSPM):     │  │  Defender for Key Vault         │ │
│  │  • Attack path analysis    │  │  Defender for Resource Manager  │ │
│  │  • Cloud security graph    │  │  Defender for DNS               │ │
│  │  • Agentless scanning      │  │  Defender for APIs              │ │
│  │  • Governance rules        │  │                                 │ │
│  │  • Data-aware security     │  │  Each plan = per-resource cost  │ │
│  │  • External attack surface │  │  Enable per subscription        │ │
│  └────────────────────────────┘  └─────────────────────────────────┘ │
│                                                                      │
│  Multi-cloud support:                                                │
│  • AWS: native connector (CloudTrail, Security Hub, EKS, EC2)       │
│  • GCP: native connector (Security Command Center, GKE, Compute)    │
│  • On-prem: Azure Arc-enabled servers                                │
└──────────────────────────────────────────────────────────────────────┘
```

> 🗓️ **Plan-catalogue currency note (2026).** Microsoft stopped accepting *new* onboarding for a few
> standalone plans: **Defender for DNS** is now delivered inside **Defender for Servers**; **Defender for Key Vault**
> and **Defender for Resource Manager** moved to a fixed pricing model; **Defender for Kubernetes** and **Defender
> for Container Registry** were long ago replaced by **Defender for Containers**. Existing subscriptions keep their
> protection, and exam material still names all of these plans - so learn the names, but do not design a *new*
> environment around "enable the Defender for DNS plan".

### Key architect decision: CSPM vs CWPP

| Feature | Free CSPM | Defender CSPM (paid) | CWPP (Defender plans) |
|---------|-----------|---------------------|---------------------|
| Purpose | Posture visibility | Advanced posture + attack paths | Runtime protection |
| Scope | Azure only | Azure + AWS + GCP | Per workload type |
| Detection | Config-based | Config + data + attack paths | Runtime threats |
| Example | "Storage account allows public access" | "This public storage leads to a path to your SQL database" | "Suspicious process on server" |
| Cost | Free | Per-server/month | Per-resource/month |

## Bad practice → Best practice (posture & endpoints)

Quick-reference table. Every row below maps to something the SC-100 exam will probe.

| ❌ Bad / legacy | ✅ Best practice | Why |
|-----------------|------------------|-----|
| Free CSPM only, no Defender plans anywhere | **Defender CSPM on all subs + targeted CWPP plans on prod** | Free tier has no attack paths and no runtime protection |
| Enable every Defender plan on every subscription | **Tier by environment** (prod = full, dev = Servers P1, sandbox = free CSPM) | Defender plans are per-resource; cost explodes if you spray |
| Each cloud has its own security console | **One Defender for Cloud pane** via AWS/GCP native connectors | Single backlog, cross-cloud attack paths |
| Deploy Azure Monitor Agent manually on on-prem VMs | **Azure Arc first**, then Defender/Policy deploy via Arc | Arc is the adapter; without it, policy/Defender can't target the VM |
| Same local admin password everywhere | **Windows LAPS** (Entra ID or AD DS backed) | Stops one compromised box from unlocking all others |
| Install EDR on MRI machines / PLCs | **Defender for IoT passive network sensor** | You can't put agents on medical/industrial devices |
| Patch servers manually, quarterly | **Azure Update Manager** (Arc-enabled for on-prem) | Centralized schedules, compliance-ready |
| Treat Secure Score as the only KPI | **Secure Score + MCSB compliance + attack-path count** | Score is coarse; MCSB maps to audit controls |
| Leave forgotten dev subdomains on the internet | **EASM scan + quarterly cleanup review** | Attackers find them first (shadow IT is a top breach vector) |
| Keep RDP/SSH open on VMs | **Azure Bastion + JIT VM access** | No exposed management ports |
| Trust "it's internal, so it's safe" | **Assume breach**: segment, log, Defender everywhere including dev | Dev is routinely used as a pivot |
| Report posture as one Secure Score number | **Microsoft Security Exposure Management**: attack paths, attack surface map, and *initiatives* scoped to a threat (ransomware, identity) | A named SC-100 objective; it answers "how exposed are we to X?", which Secure Score cannot |


In [ ]:
import json

# ===================================================================
# SCENARIO: Design Defender for Cloud strategy for Litware Inc
# ===================================================================

SCENARIO = """
COMPANY: Litware Inc
EMPLOYEES: 8,000
INDUSTRY: Healthcare (HIPAA compliance required)

INFRASTRUCTURE:
  Azure:
    - 5 subscriptions (prod, staging, dev, shared-services, sandbox)
    - 150 VMs (Windows and Linux)
    - 20 AKS clusters
    - 50 Azure SQL databases with PHI (Protected Health Information)
    - 30 Storage accounts
    - 10 App Services
    - Key Vaults for secrets management

  AWS (legacy):
    - 2 accounts (50 EC2 instances, 10 RDS databases)
    - Plan to migrate 80% to Azure over 18 months

  On-premises:
    - 100 servers in hospital data centers
    - Medical IoT devices (MRI machines, patient monitors)
    - OT network for building management systems

CURRENT STATE:
  - Defender for Cloud enabled (free tier) on Azure subscriptions
  - Secure Score: 35/100
  - No Defender plans enabled
  - No coverage for AWS or on-prem
  - No Azure Arc deployment

BUDGET: Can enable paid features on production workloads only
"""

print(SCENARIO)

In [ ]:
# ===================================================================
# DESIGN: Which Defender plans to enable and where
# ===================================================================

DEFENDER_PLAN_DESIGN = [
    {
        'plan': 'Defender CSPM',
        'scope': 'All subscriptions + AWS accounts',
        'justification': 'Attack path analysis critical for HIPAA — find paths to PHI data. Multi-cloud visibility for AWS migration.',
        'priority': 'Critical',
        'cost_impact': 'Per-server pricing, moderate cost',
    },
    {
        'plan': 'Defender for Servers P2',
        'scope': 'Production VMs (Azure + AWS via Arc) + on-prem servers (via Arc)',
        'justification': 'P2 adds agentless + agent-based vulnerability assessment, file integrity monitoring, JIT VM access and agentless malware/secret scanning. Healthcare servers are high-value targets. (Adaptive application controls and adaptive network hardening were retired in 2024 with the MMA agent - do not design around them.)',
        'priority': 'Critical',
        'cost_impact': 'Highest cost item — consider P1 for dev/staging',
    },
    {
        'plan': 'Defender for Containers',
        'scope': 'All AKS clusters (20)',
        'justification': 'Runtime protection for containers, image vulnerability scanning, admission control.',
        'priority': 'High',
        'cost_impact': 'Per-core pricing',
    },
    {
        'plan': 'Defender for Databases',
        'scope': 'All Azure SQL databases (50) + AWS RDS (10)',
        'justification': 'PHI data in databases — need threat detection for SQL injection, anomalous access, data exfiltration.',
        'priority': 'Critical',
        'cost_impact': 'Per-server pricing',
    },
    {
        'plan': 'Defender for Storage',
        'scope': 'Production storage accounts (15 of 30)',
        'justification': 'Malware scanning for uploaded files, anomalous access detection. Critical for PHI documents.',
        'priority': 'High',
        'cost_impact': 'Per-transaction + per-GB scanned',
    },
    {
        'plan': 'Defender for Key Vault',
        'scope': 'All Key Vaults',
        'justification': 'Detect unusual access to secrets, keys, and certificates. Low cost, high value. Note: this plan moved to a fixed pricing model in 2026 and no longer accepts new per-transaction onboarding.',
        'priority': 'High',
        'cost_impact': 'Low — per-transaction pricing',
    },
    {
        'plan': 'Defender for App Service',
        'scope': 'Production App Services (5 of 10)',
        'justification': 'Detect web application attacks, dangling DNS, suspicious activity.',
        'priority': 'Medium',
        'cost_impact': 'Per-app pricing',
    },
    {
        'plan': 'Defender for IoT',
        'scope': 'Medical IoT + OT networks (separate deployment)',
        'justification': 'OT/IoT devices cannot run agents — agentless network monitoring. Critical for patient safety.',
        'priority': 'Critical',
        'cost_impact': 'Per-device pricing — may need separate budget approval',
    },
]

print('=== Defender for Cloud Plan Design ===\n')
print(f'{"Plan":<30} {"Scope":<45} {"Priority":<12} {"Cost Impact"}')
print('─' * 120)
for plan in DEFENDER_PLAN_DESIGN:
    print(f'{plan["plan"]:<30} {plan["scope"]:<45} {plan["priority"]:<12} {plan["cost_impact"]}')

print('\nDesign principle: Production workloads get full coverage.')
print('Dev/staging get Defender for Servers P1 (cheaper) or no CWPP.')
print('Sandbox subscriptions: free CSPM only — no paid plans.')

## Azure Arc — Extending Azure Management to Hybrid/Multicloud

```
                    ┌─────────────────────────────────┐
                    │         AZURE ARC                │
                    │   "Extend Azure to anywhere"     │
                    └─────────────┬───────────────────┘
                                  │
          ┌────────────────────────┼────────────────────────┐
          │                        │                        │
   ┌──────▼──────┐         ┌───────▼──────┐         ┌──────▼──────┐
   │ Arc-enabled  │         │ Arc-enabled   │         │ Arc-enabled  │
   │ Servers      │         │ Kubernetes    │         │ Data Svcs    │
   │              │         │               │         │              │
   │ On-prem VMs  │         │ EKS, GKE,     │         │ SQL Managed  │
   │ AWS EC2      │         │ on-prem K8s   │         │ Instance     │
   │ GCP Compute  │         │               │         │ PostgreSQL   │
   │              │         │ Benefits:     │         │              │
   │ Benefits:    │         │ • GitOps      │         │ Benefits:    │
   │ • Azure Pol  │         │ • Monitoring  │         │ • Azure mgmt│
   │ • Defender   │         │ • Defender    │         │ • Elastic    │
   │ • Monitor    │         │ • Policy      │         │ • HA/DR     │
   │ • Update Mgmt│         │               │         │              │
   │ • SSH access │         │               │         │              │
   └──────────────┘         └───────────────┘         └──────────────┘
```

### When Arc is required (exam hot topic):

| Scenario | Arc needed? | Why |
|----------|------------|-----|
| Apply Azure Policy to AWS EC2 | Yes | Policy needs Arc agent to evaluate |
| Defender for Servers on on-prem | Yes | Defender uses Arc as deployment mechanism |
| Monitor GCP VMs with Azure Monitor | Yes | AMA agent deployed via Arc |
| Defender CSPM for AWS accounts | No | Native AWS connector (agentless) |
| Defender for Cloud Apps (SaaS) | No | Cloud-to-cloud integration |
| Azure Update Manager for on-prem | Yes | Requires Arc-enabled servers |

In [ ]:
# ===================================================================
# EASM (External Attack Surface Management)
# Discovers assets exposed to the internet that you may not know about
# ===================================================================

EASM_DESIGN = {
    'What EASM discovers': [
        'Domains and subdomains (including forgotten/shadow IT)',
        'IP addresses and ASN ranges',
        'Web applications and APIs',
        'SSL certificates (expired, weak)',
        'Open ports and services',
        'Known vulnerabilities (CVEs) on exposed assets',
        'Third-party hosted infrastructure',
    ],
    'Architecture decisions': [
        'Deploy EASM in the Defender for Cloud environment',
        'Seed discovery with: primary domains, known IP ranges, ASN numbers',
        'Integrate findings with Defender CSPM attack path analysis',
        'Set up alerts for: new unknown subdomains, expired certificates, high-severity CVEs',
        'Review cadence: weekly automated scan, monthly manual review',
    ],
    'Common findings in healthcare': [
        'Patient portal running outdated TLS 1.0',
        'Forgotten dev/staging environments exposed to internet',
        'IoT device management consoles with default credentials',
        'Third-party vendor portals with Litware branding (shadow IT)',
        'Expired SSL certificates on public-facing apps',
    ],
}

print('=== External Attack Surface Management (EASM) ===\n')
for category, items in EASM_DESIGN.items():
    print(f'\n{category}:')
    for item in items:
        print(f'  • {item}')

In [ ]:
# ===================================================================
# ENDPOINT SECURITY DESIGN
# Different endpoint types need different security approaches
# ===================================================================

ENDPOINT_STRATEGY = [
    {
        'type': 'Corporate Windows desktops',
        'count': '5,000',
        'solution': 'Defender for Endpoint P2 via Intune',
        'management': 'Intune MDM with compliance policies',
        'key_features': ['EDR', 'Attack surface reduction rules', 'Automated investigation', 'Device compliance for CA'],
        'baseline': 'Microsoft security baseline for Windows 11',
    },
    {
        'type': 'Windows servers (Azure + on-prem)',
        'count': '250',
        'solution': 'Defender for Servers P2 (includes Defender for Endpoint)',
        'management': 'Azure Arc (on-prem) + Azure native (cloud)',
        'key_features': ['Vulnerability assessment (agentless + agent)', 'File integrity monitoring', 'JIT VM access', 'Agentless secret and malware scanning'],
        'baseline': 'MCSB (Microsoft Cloud Security Benchmark)',
    },
    {
        'type': 'Linux servers',
        'count': '50',
        'solution': 'Defender for Endpoint on Linux + Defender for Servers P1',
        'management': 'Azure Arc + Ansible for configuration',
        'key_features': ['Antimalware', 'EDR', 'Vulnerability assessment'],
        'baseline': 'CIS benchmark for Ubuntu/RHEL',
    },
    {
        'type': 'Mobile devices (iOS/Android)',
        'count': '3,000',
        'solution': 'Defender for Endpoint mobile + Intune MAM',
        'management': 'Intune MAM (BYOD) or MDM (corporate-owned)',
        'key_features': ['Phishing protection', 'Jailbreak detection', 'App protection policies', 'Web content filtering'],
        'baseline': 'Intune app protection policy (Level 2 - Enterprise)',
    },
    {
        'type': 'Medical IoT devices',
        'count': '500+',
        'solution': 'Defender for IoT (agentless network monitoring)',
        'management': 'Network sensor + Defender for IoT console',
        'key_features': ['Asset discovery', 'Vulnerability detection', 'Anomalous behavior', 'Network segmentation validation'],
        'baseline': 'IEC 62443 for medical devices',
    },
    {
        'type': 'OT/Building management',
        'count': '100+',
        'solution': 'Defender for IoT (OT sensor)',
        'management': 'Air-gapped or connected OT sensor',
        'key_features': ['Protocol detection (Modbus, BACnet)', 'Firmware vulnerability', 'PLC monitoring'],
        'baseline': 'NIST SP 800-82 for ICS/SCADA',
    },
]

print('=== Endpoint Security Strategy ===\n')
print(f'{"Endpoint Type":<30} {"Count":<8} {"Solution":<45} {"Baseline"}')
print('─' * 130)
for ep in ENDPOINT_STRATEGY:
    print(f'{ep["type"]:<30} {ep["count"]:<8} {ep["solution"]:<45} {ep["baseline"]}')

In [ ]:
# ===================================================================
# WINDOWS LAPS DESIGN
# Local Administrator Password Solution — randomizes local admin passwords
# ===================================================================

LAPS_DESIGN = {
    'What it solves': 'Same local admin password across all devices = lateral movement goldmine',
    'How it works': 'Automatically rotates local admin passwords, stores them in Entra ID or AD DS',
    'Architecture options': {
        'Cloud-native (Entra ID LAPS)': {
            'best_for': 'Entra ID joined or Entra hybrid joined devices',
            'password_storage': 'Entra ID device object',
            'retrieval': 'Azure Portal, Graph API, Intune',
            'rotation': 'Configurable via Intune policy (default: 30 days)',
        },
        'On-premises (AD DS LAPS)': {
            'best_for': 'Domain-joined servers and workstations not managed by Intune',
            'password_storage': 'AD DS computer object (encrypted attribute)',
            'retrieval': 'PowerShell, LAPS UI, delegated AD permissions',
            'rotation': 'Group Policy managed',
        },
    },
}

print('=== Windows LAPS Architecture ===\n')
print(f'Problem: {LAPS_DESIGN["What it solves"]}')
print(f'Solution: {LAPS_DESIGN["How it works"]}\n')

for option, details in LAPS_DESIGN['Architecture options'].items():
    print(f'\n--- {option} ---')
    for key, value in details.items():
        print(f'  {key}: {value}')

print('\nDesign recommendation for Litware:')
print('  • Entra ID LAPS for all Intune-managed devices (desktops, laptops)')
print('  • AD DS LAPS for on-prem servers not yet Arc-enabled')
print('  • Both can coexist — device joins determine which backs up where')

In [ ]:
# ===================================================================
# PRACTICAL EXAMPLE: Posture maturity scorer
# A tiny function you can adapt to rank your own subscriptions.
# Given a "current state" dict, it tells you the next 3 steps on the
# CSPM -> CWPP -> attack-path maturity ladder.
# ===================================================================

LADDER = [
    ('free_cspm_on',         'Turn on Defender for Cloud free tier (Secure Score)'),
    ('mcsb_policy_assigned', 'Assign the MCSB policy initiative at management-group scope'),
    ('defender_cspm_on',     'Enable Defender CSPM for attack path analysis'),
    ('servers_p1_on',        'Enable Defender for Servers P1 on production VMs'),
    ('servers_p2_on',        'Upgrade prod VMs to Servers P2 (VA, FIM, JIT)'),
    ('sql_plan_on',          'Enable Defender for Databases (SQL/RDS threat detection)'),
    ('storage_plan_on',      'Enable Defender for Storage (malware scan)'),
    ('keyvault_plan_on',     'Enable Defender for Key Vault (cheap, high value)'),
    ('containers_plan_on',   'Enable Defender for Containers on AKS'),
    ('arc_on_onprem',        'Onboard on-prem + AWS servers to Azure Arc'),
    ('iot_sensor_on',        'Deploy Defender for IoT sensor for OT/medical devices'),
    ('easm_enabled',         'Turn on EASM and add primary domains / IP ranges'),
]

def next_steps(state: dict, top_n: int = 3):
    missing = [(k, desc) for k, desc in LADDER if not state.get(k)]
    return missing[:top_n]

# Example: Litware production subscription, partially hardened
litware_prod = {
    'free_cspm_on': True,
    'mcsb_policy_assigned': True,
    'servers_p1_on': True,
}

done = sum(1 for k, _ in LADDER if litware_prod.get(k))
print(f'Maturity: {done}/{len(LADDER)} rungs complete\n')
print('Next 3 recommended steps:')
for key, desc in next_steps(litware_prod, 3):
    print(f'  [ ] {desc}   (flag: {key})')


In [ ]:
# ===================================================================
# ARCHITECTURE QUIZ: Posture & Endpoints
# ===================================================================

QUIZ = [
    {
        'question': 'Litware needs to identify attack paths from an internet-facing web app to their\n'
                    'Azure SQL database containing PHI. Which capability provides this?',
        'options': {
            'A': 'Defender for Cloud Secure Score',
            'B': 'Defender CSPM attack path analysis',
            'C': 'Azure Policy compliance dashboard',
            'D': 'Microsoft Sentinel threat hunting',
        },
        'answer': 'B',
        'explanation': 'Defender CSPM\'s attack path analysis uses the cloud security graph to find paths '
                       'like: Internet → Web App (with vulnerability) → Managed Identity → SQL Database (with PHI). '
                       'Secure Score gives overall posture but not attack paths. Azure Policy checks configurations '
                       'but doesn\'t correlate them into attack chains.',
    },
    {
        'question': 'Litware has 50 EC2 instances in AWS that need Azure Policy evaluation and\n'
                    'Defender for Servers protection. What must they deploy first?',
        'options': {
            'A': 'AWS native connector in Defender for Cloud',
            'B': 'Azure Arc agent on each EC2 instance',
            'C': 'Azure Monitor Agent (AMA) directly on EC2',
            'D': 'AWS Systems Manager integration',
        },
        'answer': 'B',
        'explanation': 'Azure Arc is the prerequisite for applying Azure Policy and deploying Defender for '
                       'Servers on non-Azure machines. The Arc agent makes EC2 instances appear as Azure '
                       'resources, enabling policy evaluation, Defender protection, and Azure Monitor. '
                       'The AWS native connector provides CSPM (agentless) but not per-server protection.',
    },
    {
        'question': 'Litware has 500 medical IoT devices (MRI machines, patient monitors).\n'
                    'Which approach is BEST for securing these devices?',
        'options': {
            'A': 'Deploy Defender for Endpoint agent on each device',
            'B': 'Deploy Defender for IoT with network sensors (agentless)',
            'C': 'Connect devices to Azure Arc for policy management',
            'D': 'Use Intune MDM to manage the devices',
        },
        'answer': 'B',
        'explanation': 'Medical IoT devices cannot run agents — they have specialized firmware and installing '
                       'software could void warranties or affect patient safety. Defender for IoT uses passive '
                       'network sensors that monitor traffic without touching the devices. This provides asset '
                       'discovery, vulnerability detection, and anomaly detection without any device modification.',
    },
]

print('=== Posture & Endpoint Architecture Quiz ===\n')
for i, q in enumerate(QUIZ, 1):
    print(f'Question {i}:')
    print(f'{q["question"]}\n')
    for key, option in q['options'].items():
        marker = '>>>' if key == q['answer'] else '   '
        print(f'  {marker} {key}. {option}')
    print(f'\n  Answer: {q["answer"]}')
    print(f'  Why: {q["explanation"]}')
    print()